## Topic: Conditional Chain in LangChain

### Agenda

- 1. Introduction of conditional chain

- 2. Comparison Three Chain Patterns (sequential, parallel, conditional) 

- 3. Example of conditional chain

- 4. summary 


### 1. Introduction of conditional chain

- Definition:
    - A Conditional Chain chooses which path to execute based on a condition or the input/result.

    - Only the appropriate branch is executed.


- for Conditional chain use
    - from langchain_core.runnables import RunnableBranch

In [ ]:
""" 
        - syntax of RunnableBranch

from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (condition_1, chain_1),
    (condition_2, chain_2),
    default_chain
)

- condition determines whether chain_1 should run.
    - If condition 1 is True →  chain_1.
    - Otherwise → Chain_2


"""

# The RunnableBranch is similar to if-elif-else condition statement
"""  
if condition_1:
    run chain_1

elif condition_2:
    run chain_2

else:
    run default_chain

"""

### 2. Comparison Three Chain Patterns (sequential, parallel, conditional) 


In [ ]:
"""  - only conditional chain 
Input
  ↓
Condition
  ↓
┌───────────────┐
│               │
YES             NO
│               │
↓               ↓
Chain A       Chain B
- 
"""

In [ ]:
""" 
┌──────────────────────────────────────────────────┐
│                 LANGCHAIN CHAINS                 │
└──────────────────────────────────────────────────┘

1. SEQUENTIAL
   A → B → C

   Used when:
   Output of A is required by B.


2. PARALLEL
          ┌→ A
   Input ─┼→ B
          └→ C

   Used when:
   A, B, and C are independent.


3. CONDITIONAL
              ┌→ A
   Input → Decision
              └→ B

   Used when:
   The next step depends on a condition.

"""

### 3. Example of conditional chain

- Idea:
    - prompt1: user give an a {feedback}

    - LLM1:
        - input: prompt1
        - process: find the sentiment of the prompt1
        - response1: Positive or Negative (classify the sentiment)

    - IF sentiment == Positive:
        - prompt2 : for positive relative 
        
        - LLM2:
            - input: prompt2
            - process: give the appropriate positive message
            - response2: replay Positive message ( give a 5-star review)

    - ElIF sentiment == Negative:
        - prompt3 : for Negative relative 
        
        - LLM3:
            - input: prompt3
            - process: give the appropriate Negative message
            - response2: replay Negative message (gmail the customer support)

    


In [ ]:
# example 1:
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

from langchain_core.runnables import RunnableBranch, RunnableLambda

from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

model = ChatOpenAI()

# parser 1 : Output string formatting
parser = StrOutputParser()

# parser 2 : for validation and output formatting of the llm 
class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(
        description='Give the sentiment of the feedback'
        )

# parser 1 object 
parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)


# classifier chain for sentiment (Positive or Negative) 
classifier_chain = prompt1 | model | parser2

# Prompt2 : for Positive sentiment
prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

# Prompt3 : for Negative sentiment
prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

# apply RunnableBranch for conditional chaining 
branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

# final chain -> the complete chain path of which one is execute 
chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))

chain.get_graph().print_ascii()

### 4. summary 

- Conditional Chain
    - A Conditional Chain is a LangChain workflow that dynamically selects which chain or operation to execute based on a condition. It is useful for routing different types of inputs to specialized processing workflows. In modern LangChain, RunnableBranch can be used to implement conditional routing.


- Key takeaway:
    - Sequential Chain = Dependency
    - Parallel Chain = Independence
    - Conditional Chain = Decision